In [9]:
import subprocess
import os
import pandas as pd
import numpy as np
from core.reader import read_ansys_csv, load_experimental_data

ANSYS_EXE_PATH = r"D:\Program Files\ANSYS Inc\ANSYS Student\v252\ansys\bin\winx64\MAPDL.exe" 
WORKING_DIR = os.getcwd()

def run_ansys_simulation(params):
    with open('voce_hardening_params.txt', 'w') as f:
        for p in params:
            f.write(f"{p}\n")

    input_file = "voce_hardening.mac"
    output_file = "ansys.out"
    
    cmd = [
        ANSYS_EXE_PATH, 
        "-b",
        "-j", "opt_run",
        "-dir", WORKING_DIR, 
        "-i", input_file, 
        "-o", output_file
    ]

    try:
        subprocess.run(cmd, check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        print("Ошибка ANSYS:", e)
        return None

    try:
        df_res = read_ansys_csv("voce_hardening.csv")
        return df_res
    except Exception as e:
        print(f"Ошибка чтения CSV: {e}")
        return None

In [10]:
real_experiment_data_folder = r'D:\Users\complex_deformations\P29\experimental_vint'
df_exp = load_experimental_data(real_experiment_data_folder)

zero_row = pd.DataFrame(0.0, columns=df_exp.columns, index=[0])
zero_row['Time'] = 1.0
df_exp = pd.concat([zero_row, df_exp]).reset_index(drop=True)

def objective_function(params):
    """
    Считает ошибку между экспериментом и моделью Шабоша.
    params: [sig_y, c1, g1, c2, g2, c3, g3, R_inf, b]
    """
    print(f"Simulating: {params}")
    
    df_ansys = run_ansys_simulation(params)
    
    if df_ansys is None or len(df_ansys) != len(df_exp):
        return 1e9
    
    # Считаем MSE по компонентам напряжений
    mse_zz = np.mean((df_ansys['S_ZZ'] - df_exp['S_ZZ'])**2)
    mse_tt = np.mean((df_ansys['S_TT'] - df_exp['S_TT'])**2)
    mse_tz = np.mean((df_ansys['S_TZ'] - df_exp['S_TZ'])**2)
    
    total_error = mse_zz + mse_tt + mse_tz
    print(f"Error: {total_error:.2f}")
    return total_error

In [11]:
import psutil

def kill_ansys_processes():
    for proc in psutil.process_iter():
        if proc.name() in ['ANSYS.exe', 'MAPDL.exe', 'ansys.exe']:
            proc.kill()

kill_ansys_processes()

In [ ]:
from scipy.optimize import minimize

x0 = [
    216.49263744583521,    # Sig_Y
    120265.7074049233,     # C1
    623.4456252215341,     # gamma1
    7828.078591604883,     # C2
    46.929596510667835,    # gamma2
    1073.0776477224936,    # C3
    50.006332654928684,    # gamma3
    100.0,                 # R_inf
    10.0,                  # b          
]
print(f"Запуск локальной оптимизации (Nelder-Mead) с начальной точки:\n{x0}")

bounds = [
    (200, 400),           # Sig_Y
    (1e4, 5e5),           # C1
    (100, 5000),          # gamma1
    (1e3, 5e4),           # C2
    (10, 500),            # gamma2
    (100, 1e4),           # C3
    (0, 100),             # gamma3
    (0, 300),             # R_inf
    (0.1, 100)            # b

]

res = minimize(
    objective_function, 
    x0, 
    method='Nelder-Mead',
    bounds=bounds,
    options={
        'maxiter': 200,    # Количество запусков ANSYS
        'disp': True,
        'xatol': 1.0,     # Точность поиска
        'fatol': 100.0    # Точность по функции ошибки
    }
)

print("\nОптимизация завершена!")
print("Лучшие параметры:", res.x)
print("Лучшая ошибка:", res.fun)

In [ ]:
# x0 = [
#     216.49263744583521,    # Sig_Y
#     120265.7074049233,     # C1
#     623.4456252215341,     # gamma1
#     7828.078591604883,     # C2
#     46.929596510667835,    # gamma2
#     1073.0776477224936,    # C3
#     50.006332654928684,    # gamma3
#     100.0,                 # R_inf
#     10.0,                  # b
# ]

# initial_error = objective_function(x0)
# print(f"Ошибка на старте: {initial_error}")

Simulating: [216.49263744583521, 120265.7074049233, 623.4456252215341, 7828.078591604883, 46.929596510667835, 1073.0776477224936, 50.006332654928684, 100.0, 10.0]
Error: 115530.99
Ошибка на старте: 115530.99123398421
